# Laboratorio 4: Costruisci un grafo a mano

---

# Lab 4: Build a graph by hand

**Come si usa.** Esegui le celle dall'alto in basso (Maiusc+Invio). Serve un account Google gratuito
per usare Colab. Il testo completo del laboratorio e' sul sito: https://corso-rag.giuseppebosi.com/moduli/04-dal-recupero-al-grafo/LAB/

---

**How to use it.** Run the cells from top to bottom (Shift+Enter). Colab needs a free Google account.
The full text of the lab is on the site: https://corso-rag.giuseppebosi.com/en/moduli/04-dal-recupero-al-grafo/LAB/

In [ ]:
# Prepara l'ambiente / Set up the environment (about 1 minute)
import os, sys, subprocess
if sys.version_info[:2] not in [(3, 11), (3, 12)]:
    print("ATTENZIONE: questo corso e' provato con Python 3.11 o 3.12; hai "
          f"{sys.version_info.major}.{sys.version_info.minor}. "
          "WARNING: the course is tested on Python 3.11 or 3.12.")
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
REPO_DIR = os.path.join(BASE, "rag-chatbot-course")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/JoeBosi/rag-chatbot-course.git", REPO_DIR], check=True)
os.chdir(os.path.join(REPO_DIR, "corso"))
!pip install -q openai python-dotenv httpx PyYAML pypdf fastembed numpy transformers
print("Cartella di lavoro / working folder:", os.getcwd())

## La chiave gratuita / The free key

Questo laboratorio chiama un modello: serve una chiave **Groq**, gratuita e senza carta di credito
([CHIAVI.md](https://corso-rag.giuseppebosi.com/CHIAVI/) spiega come ottenerla in due minuti).
Poi in Colab: icona della **chiave** nella barra a sinistra, **Aggiungi nuovo segreto**, nome
`GROQ_API_KEY`, valore la tua chiave, e attiva **Accesso da notebook**. Se non la imposti, la cella
te la chiede al momento. La chiave non finisce mai nel notebook.

---

This lab calls a model, so you need a free **Groq** key, no credit card
([CHIAVI.md](https://corso-rag.giuseppebosi.com/en/CHIAVI/) shows how to get one in two minutes).
Then in Colab: the **key** icon in the left bar, **Add new secret**, name `GROQ_API_KEY`, value your
key, and switch on **Notebook access**. If you skip this, the cell asks for the key when you run it.
The key is never saved in the notebook.

In [ ]:
# La chiave Groq / The Groq key. Mai scritta nel notebook / never written in the notebook.
import os
from getpass import getpass

def leggi_chiave(nome="GROQ_API_KEY"):
    if os.environ.get(nome):
        return os.environ[nome]
    try:
        from google.colab import userdata          # Colab: icona della chiave, a sinistra
        return userdata.get(nome)
    except Exception:
        pass
    try:
        return getpass(f"Incolla {nome} / paste {nome}: ")
    except Exception:
        return ""

chiave = leggi_chiave()
if chiave:
    os.environ["GROQ_API_KEY"] = chiave
    os.environ["LLM_PROVIDER"] = "groq"
    print("Chiave caricata (non viene mostrata) / key loaded (not displayed).")
else:
    print("Nessuna chiave. I passaggi che chiamano il modello si fermeranno con "
          "il messaggio 'Manca GROQ_API_KEY'. Vedi la cella sopra.\n"
          "No key. Steps that call the model will stop with 'Manca GROQ_API_KEY'. See the cell above.")
![ -f .env ] || cp .env.example .env

Le celle con un'etichetta **[serve la chiave]** chiamano il modello. Senza chiave si fermano con
`Manca GROQ_API_KEY in .env`: non e' un guasto, inserisci la chiave come spiegato sopra e rilancia.
Le altre celle funzionano anche senza.

---

Cells labelled **[needs the key]** call the model. Without a key they stop with
`Manca GROQ_API_KEY in .env` (the message is Italian): this is not a fault, add the key as explained
above and run the cell again. The other cells work without it.

Costo in quota: circa 30.000 token per costruire tutto il grafo, cioe' quattro o cinque minuti di lavoro piu' le attese. La cella di costruzione e' l'unica costosa.

---

Quota cost: about 30,000 tokens to build the whole graph, i.e. four or five minutes of work plus waiting. The build cell is the only expensive one.

## Starter o soluzione? / Starter or solution?

Nel laboratorio scrivi tu alcune funzioni (i TODO). Qui sotto scegli:
`USA_SOLUZIONE = True` copia la soluzione, cosi' tutte le celle girano subito;
`USA_SOLUZIONE = False` copia lo **starter** con i TODO: apri il file nel pannello **File** a sinistra
(cartella `rag-chatbot-course/corso/memoria`), scrivi la funzione, salva e rilancia le celle.

---

In the lab you write some functions yourself (the TODOs). Choose below:
`USA_SOLUZIONE = True` copies the solution so every cell runs straight away;
`USA_SOLUZIONE = False` copies the **starter** with the TODOs: open the file in the **Files** panel on
the left (folder `rag-chatbot-course/corso/memoria`), write the function, save, and run the cells again.

## Passo 1: vedi il fallimento **[serve la chiave]**

Guarda quali pezzi recupera: nessuno parla della caldaia.

---

## Step 1: see the failure **[needs the key]**

Look at which chunks it retrieves: none mentions the boiler.

In [ ]:
USA_SOLUZIONE = True
MOD = "moduli/04-dal-recupero-al-grafo"
ORIGINE = "soluzione" if USA_SOLUZIONE else "starter"
!python $MOD/starter/prova_multihop.py

## Passo 2: la pazienza

Deve stampare `2.375` e `8`.

---

## Step 2: patience

It must print `2.375` and `8`.

In [ ]:
!cp $MOD/$ORIGINE/pazienza.py memoria/
!cp $MOD/$ORIGINE/grafo.py memoria/

In [ ]:
import sys; sys.path.insert(0, "memoria")
from pazienza import secondi_da_aspettare
class Finto(Exception): pass
print(secondi_da_aspettare(Finto("Rate limit reached. Please try again in 1.875s"), 0))
print(secondi_da_aspettare(Finto("errore generico"), 3))

## Passo 3: l'estrazione **[serve la chiave]**

Prima su un solo documento (una chiamata): entita' con `nome` e `tipo`, relazioni `da / relazione / a`.

---

## Step 3: extraction **[needs the key]**

First on a single document (one call): entities with `nome` and `tipo`, relations `da / relazione / a`.

In [ ]:
import json
from grafo import estrai
from pathlib import Path
t = Path("progetto-finale/corpus/note/2026-03-12-caldaia-e4.md").read_text()
print(json.dumps(estrai(t), indent=2, ensure_ascii=False))

Poi tutto il corpus **[serve la chiave]**: una riga per documento, e `quota esaurita, aspetto ...` quando serve. Scrive `dati/grafo.json`.

---

Then the whole corpus **[needs the key]**: one line per document, and `quota esaurita, aspetto ...` when needed. Writes `dati/grafo.json`.

In [ ]:
!python memoria/grafo.py costruisci

## Passo 4: la visita

Il numero di relazioni deve crescere da 1 a 2 salti e poco da 2 a 3.

---

## Step 4: the traversal

The number of relations must grow from 1 to 2 hops and barely from 2 to 3.

In [ ]:
!python memoria/grafo.py mostra "caldaia"
from grafo import carica, vicini, trova_nodo
g = carica()
n = trova_nodo(g, "caldaia termovex")
for s in (1, 2, 3):
    print(s, "salti ->", len(vicini(g, n, salti=s)), "relazioni")

## Passo 5: la risposta impossibile **[serve la chiave]**

Deve rispondere **3,5 ore**, con gli addendi 1,5 e 2.

---

## Step 5: the impossible answer **[needs the key]**

It must answer **3.5 hours**, showing the addends 1.5 and 2.

In [ ]:
!python memoria/grafo.py chiedi "Quante ore ho passato in tutto sulla caldaia?"
!python memoria/grafo.py chiedi "Che pressione hanno le gomme della bici?"

## Passo 6: rompi il grafo apposta

Cerca i nodi duplicati. Il tuo risultato puo' essere diverso dal nostro, ma qualche duplicato lo trovi.

---

## Step 6: break the graph on purpose

Look for duplicate nodes. Your result may differ from ours, but you will find some duplicates.

In [ ]:
g = json.load(open("dati/grafo.json"))
for parola in ["caldaia", "lavatrice", "rasaerba", "bici"]:
    print(parola, "->", [e["nome"] for e in g["entita"] if parola in e["nome"].lower()])

## Passo 7: verifica

---

## Step 7: check

In [ ]:
!python $MOD/verifica.py